In [1]:
import os
import random
import csv
import pathlib

import numpy as np
import pandas as pd

import cv2
import keras
from keras import optimizers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Kaggle paths (do not change I/O semantics; just make paths correct for this environment)
BASE_INPUT = "/kaggle/input/aerial-cactus-identification"
TRAIN_DIR = os.path.join(BASE_INPUT, "train")
TEST_DIR = os.path.join(BASE_INPUT, "test")
TRAIN_CSV = os.path.join(BASE_INPUT, "train.csv")
SAMPLE_SUB = os.path.join(BASE_INPUT, "sample_submission.csv")

WORKING_DIR = "/kaggle/working"
OUTPUT_DIR = os.path.join(WORKING_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(
    "Train dir exists:",
    os.path.isdir(TRAIN_DIR),
    "n_files:",
    len(os.listdir(TRAIN_DIR)),
)
print(
    "Test dir exists :", os.path.isdir(TEST_DIR), "n_files:", len(os.listdir(TEST_DIR))
)
print("Train CSV exists:", os.path.isfile(TRAIN_CSV))
print("Sample sub exists:", os.path.isfile(SAMPLE_SUB))




2026-01-11 00:09:55.330460: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768090195.343948      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768090195.348073      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-11 00:09:55.364656: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Train dir exists: True n_files: 14176
Test dir exists : True n_files: 3326
Train CSV exists: True
Sample sub exists: True


In [2]:
class ImageDataGenerator_self(object):
    """
    Minimal custom generator preserved from original intent:
    yields (batch_images, batch_onehot_labels) indefinitely.
    Fixes:
      - no re-reading CSV inside generator
      - correct label lookup (id -> has_cactus)
      - numeric classes rather than strings
      - uses keras.utils.to_categorical reliably
    """

    def __init__(self):
        self.reset()

    def reset(self):
        self.images = []
        self.labels = []

    def flow_from_dir2(
        self, data_dir, data_list, label_df, classes, batch_size=32, shuffle=True
    ):
        # Precompute id->label mapping once
        if isinstance(label_df, pd.DataFrame):
            label_map = label_df.set_index("id")["has_cactus"].to_dict()
        else:
            raise ValueError(
                "label_df must be a pandas DataFrame with columns ['id','has_cactus']."
            )

        data_list = list(data_list)
        n_classes = len(classes)

        while True:
            if shuffle:
                random.shuffle(data_list)

            for img_path in data_list:
                full_path = os.path.join(data_dir, img_path)
                img = cv2.imread(full_path)
                if img is None:
                    # Skip unreadable files instead of crashing
                    continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
                self.images.append(img)

                y = int(label_map[img_path])
                # classes is expected to be [0, 1]
                class_index = classes.index(y)
                self.labels.append(keras.utils.to_categorical(class_index, n_classes))

                if len(self.images) == batch_size:
                    inputs = np.asarray(self.images, dtype=np.float32)
                    targets = np.asarray(self.labels, dtype=np.float32)
                    self.reset()
                    yield inputs, targets




In [3]:
# Load labels and create train/val split (preserve core approach: random split)
label_train = pd.read_csv(TRAIN_CSV)  # columns: id, has_cactus

data_dir = TRAIN_DIR
data_list = sorted(os.listdir(data_dir))

classes = [0, 1]
batch_size = 10
val_ratio = 0.2

train_list, val_list = [], []
for fname in data_list:
    if random.random() > val_ratio:
        train_list.append(fname)
    else:
        val_list.append(fname)

print("train_list:", len(train_list), "val_list:", len(val_list))

train_datagen = ImageDataGenerator_self()
val_datagen = ImageDataGenerator_self()



train_list: 11321 val_list: 2855


In [4]:
from keras.applications.vgg16 import VGG16
from keras.models import Model
from keras.layers import Dense, GlobalAveragePooling2D
from keras.callbacks import ModelCheckpoint

# Determine input shape from an actual image (original behavior preserved)
sample_img = cv2.imread(os.path.join(data_dir, data_list[0]))
img_height, img_width, img_channel = sample_img.shape

# Core model preserved: VGG16(include_top=False, weights=None) + GAP + Dense(1024) + softmax(2)
base_model = VGG16(
    include_top=False,
    weights=None,
    input_tensor=None,
    input_shape=(img_height, img_width, img_channel),
)

n_categories = 2
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation="relu")(x)
prediction = Dense(n_categories, activation="softmax")(x)
model = Model(inputs=base_model.input, outputs=prediction)

# Keras 3: use learning_rate instead of lr
model.compile(
    optimizer=optimizers.SGD(learning_rate=0.0001, momentum=0.9),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()



2026-01-11 00:10:01.533810: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1768090201.534271      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:85:00.0, compute capability: 8.6


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 32, 32, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 8, 8, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 8, 8, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 4, 4, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 2, 2, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 1, 1, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │         2,050 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,242,050 (58.14 MB)

 Trainable params: 15,242,050 (58.14 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Keras 3: ModelCheckpoint no longer supports 'period'; also metric keys are 'accuracy' not 'acc'
fpath = os.path.join(
    OUTPUT_DIR,
    "weights.{epoch:03d}-{loss:.4f}-{accuracy:.4f}-{val_loss:.4f}-{val_accuracy:.4f}.h5",
)
modelCheckpoint = ModelCheckpoint(
    filepath=fpath,
    monitor="loss",
    verbose=1,
    save_best_only=True,
    save_weights_only=False,
    mode="min",
)

# Keras 3: use model.fit (fit_generator removed)
steps_per_epoch = max(1, int(len(train_list) / batch_size))
validation_steps = max(1, int(len(val_list) / batch_size))

history = model.fit(
    x=train_datagen.flow_from_dir2(
        data_dir=data_dir,
        data_list=train_list,
        label_df=label_train,
        classes=classes,
        batch_size=batch_size,
        shuffle=True,
    ),
    steps_per_epoch=steps_per_epoch,
    epochs=1,
    verbose=2,
    validation_data=val_datagen.flow_from_dir2(
        data_dir=data_dir,
        data_list=val_list,
        label_df=label_train,
        classes=classes,
        batch_size=batch_size,
        shuffle=False,
    ),
    validation_steps=validation_steps,
    callbacks=[modelCheckpoint],
)



I0000 00:00:1768090203.235114      63 service.cc:148] XLA service 0x7f4734002f40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768090203.235141      63 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-11 00:10:03.259156: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1768090203.404057      63 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-01-11 00:10:03.858143: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_590', 40 bytes spill stores, 40 bytes spill loads

I0000 00:00:1768090207.005842      63 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



Epoch 1: loss improved from inf to 0.32493, saving model to /kaggle/working/output/weights.001-0.3249-0.8559-0.1574-0.9418.h5


1132/1132 - 14s - 13ms/step - accuracy: 0.8559 - loss: 0.3249 - val_accuracy: 0.9418 - val_loss: 0.1574


In [6]:
# Save model architecture for later reload (kept from original intent)
model_json_str = model.to_json()
with open(os.path.join(OUTPUT_DIR, "vgg16.json"), "w") as f:
    f.write(model_json_str)

print("Saved model json to:", os.path.join(OUTPUT_DIR, "vgg16.json"))



Saved model json to: /kaggle/working/output/vgg16.json


In [7]:
from keras.models import model_from_json

# Load latest/best checkpoint saved by ModelCheckpoint
weight_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".h5")]
if not weight_files:
    raise FileNotFoundError(
        f"No .h5 weight files found in {OUTPUT_DIR}. Training likely failed."
    )

latest_weight = max(
    weight_files,
    key=lambda w: pathlib.Path(os.path.join(OUTPUT_DIR, w)).stat().st_mtime,
)
print("Latest weight:", latest_weight)

with open(os.path.join(OUTPUT_DIR, "vgg16.json"), "r") as f:
    model = model_from_json(f.read())

model.load_weights(os.path.join(OUTPUT_DIR, latest_weight))
model.compile(
    optimizer=optimizers.SGD(learning_rate=0.0001, momentum=0.9),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()



Latest weight: weights.001-0.3249-0.8559-0.1574-0.9418.h5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 32, 32, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 8, 8, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 8, 8, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 4, 4, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 2, 2, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 1, 1, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │         2,050 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,242,050 (58.14 MB)

 Trainable params: 15,242,050 (58.14 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# Predict on test set in the exact order of sample_submission to guarantee alignment
sub = pd.read_csv(SAMPLE_SUB)  # columns: id, has_cactus
test_ids = sub["id"].tolist()

preds = np.zeros(len(test_ids), dtype=np.float32)

for i, pred_img in enumerate(test_ids):
    img_path = os.path.join(TEST_DIR, pred_img)
    img = cv2.imread(img_path)
    if img is None:
        preds[i] = 0.5
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
    img = img.reshape((1,) + img.shape)
    result = model.predict(img, verbose=0)
    preds[i] = float(result[0][1])  # probability of class 1 ("has_cactus")

sub["has_cactus"] = preds

pred_result_path = os.path.join(OUTPUT_DIR, "pred_result.csv")
sub.to_csv(pred_result_path, index=False)
print("Wrote predictions to:", pred_result_path)
print(sub.head())



Wrote predictions to: /kaggle/working/output/pred_result.csv
                                     id  has_cactus
0  09034a34de0e2015a8a28dfe18f423f6.jpg    0.554213
1  134f04305c795d6d202502c2ce3578f3.jpg    0.979195
2  41fad8d145e6c41868ce3617e30a2545.jpg    0.996240
3  35f8a11352c8d41b6231bb33d8d09f7e.jpg    0.971458
4  b77dc902b035887cbbc01920ce0e3151.jpg    0.995708


In [9]:
# Final submission in working directory with required name and format
submission_path = os.path.join(WORKING_DIR, "submission.csv")
sub.to_csv(submission_path, index=False)
print("Submission saved to:", submission_path)
print("Submission shape:", sub.shape)
print("Columns:", list(sub.columns))



Submission saved to: /kaggle/working/submission.csv
Submission shape: (3325, 2)
Columns: ['id', 'has_cactus']


In [10]:
print("Output dir files:", sorted(os.listdir(OUTPUT_DIR))[:10])
print("Working dir files:", sorted(os.listdir(WORKING_DIR))[:10])

Output dir files: ['pred_result.csv', 'vgg16.json', 'weights.001-0.3249-0.8559-0.1574-0.9418.h5']
Working dir files: ['aerial-cactus-identification', 'aerial-cactus-identification_shotaho_kernel4265908402_v12_C1.ipynb', 'output', 'submission.csv']
